[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 05](README.md)

# Modelo de offload y portabilidad

**Tema:** 05 · **Sesiones:** 22 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cuándo desplazar cómputo a un dispositivo compensa transferencia, inicialización y sincronización?


## Resultados de aprendizaje

- Separar host, device y runtime de offload.
- Modelar tiempo extremo a extremo.
- Diseñar fallback CPU con corrección equivalente.


## Modelo conceptual

Offload portable conserva una interfaz, no rendimiento idéntico entre dispositivos.

El costo total incluye descubrimiento, asignación, mapeo, transferencia, kernel y sincronización.

Una ruta CPU válida permite probar corrección sin afirmar evidencia de acelerador.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "05"
NOTEBOOK = "05_openmp_target/01_modelo_offload.ipynb"
assert (ROOT / "curso" / "notebooks" / "05_openmp_target" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Punto de equilibrio

Se compara CPU con dispositivo incluyendo transferencias.


In [ ]:
cpu_rate = 40e9
device_rate = 600e9
link_rate = 24e9
startup = 80e-6
assert min(cpu_rate, device_rate, link_rate, startup) > 0
for bytes_ in (1e5, 1e6, 1e7, 1e8, 1e9):
    work = 4 * bytes_
    cpu = work / cpu_rate
    device = startup + 2*bytes_/link_rate + work/device_rate
    print(f"bytes={bytes_:10.0f} cpu={cpu*1e3:8.3f}ms offload={device*1e3:8.3f}ms conviene={device<cpu}")


**Interpretación.** El modelo debe recalibrarse con el enlace, dispositivo y kernel reales; no incluye aún solapamiento ni reutilización de datos.


## Selección de ruta

Se hace explícita una política de fallback verificable.


In [ ]:
def execution_path(devices, requested=True):
    if requested and devices > 0: return "device"
    return "host-fallback"
assert execution_path(0) == "host-fallback"
assert execution_path(1) == "device"
for devices in (0, 1, 2): print(devices, execution_path(devices))


**Interpretación.** El informe registra la ruta usada; ejecutar fallback no demuestra soporte ni rendimiento del dispositivo.


## Práctica reproducible

1. Registrar número y tipo de dispositivos visibles.
2. Medir por separado transferencia y cómputo.
3. Comparar resultado con la misma referencia serial.


## Errores frecuentes

- Cronometrar solo el kernel y llamarlo tiempo total.
- Suponer que portabilidad implica ausencia de ajustes.
- Ocultar que se ejecutó fallback CPU.

## Criterios de aceptación

- Ruta host/device registrada.
- Tiempo extremo a extremo y tiempo de kernel separados.
- Tolerancia de corrección idéntica.


## Referencias y material relacionado

- [Toolchain](../../../config/course-toolchain.cmake)
- [Planeación offload](../../../docs/PLANEACION_CURSO.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 05](README.md)
